# 01 - Gaussian Curve Reconstruction

This notebook confirms the forward model that every curve-space loss term is
built on: `Loss.reconstruct_gaussians`. The model predicts, per pixel, a set
of `(amplitude, mu, sigma)` triplets. The reconstruction expands those
parameters into an elevation profile sampled on `x_axis`. All curve losses
(`mse_curve`, `l1_curve`, SSIM, coherence, the physics terms) compare these
reconstructed profiles, so we verify the expansion is correct by eye.

Modules exercised: `pipelines.backbone_pipeline.loss.Loss.reconstruct_gaussians`,
`configuration.training_config.GaussianConfig`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


## Build a minimal `Loss` instance

We do not need a trainer or a dataset. A `Loss` object only needs an
`x_axis`, a null logger/tracker, a `GaussianConfig`, and an identity
normaliser stub (so denormalisation is a no-op and we control the physical
parameters directly).

In [ ]:
from configuration.training_config import GaussianConfig, GeometryConfig, LossConfig
from pipelines.backbone_pipeline.loss import Loss
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX, N_SAMPLES = -10.0, 40.0, 128
x_axis = torch.linspace(X_MIN, X_MAX, N_SAMPLES, dtype=torch.float32)

gaussian_cfg = GaussianConfig(n_default_gaussians=2, x_min=X_MIN, x_max=X_MAX)
loss_cfg     = LossConfig(use_mse_curve=True, weight_mse_curve=1.0)

criterion = Loss(
    x_axis       = x_axis,
    logger       = NullLogger(),
    tracker      = NullTracker(),
    gaussian_cfg = gaussian_cfg,
    loss_cfg     = loss_cfg,
    norm_stats   = IdentityNormStats(),
    geometry_cfg = GeometryConfig(),
)
print('sample points :', criterion.x_axis.shape[0])
print('dx            :', round(criterion.dx, 4))


## Reconstruct a single two-component profile

Parameters are laid out as `(amp, mu, sigma)` per Gaussian along the channel
axis. We place one narrow peak and one broad peak at known centres and check
that the reconstructed curve has peaks at those centres with the expected
heights and widths.

In [ ]:
amp1, mu1, sig1 = 3.0,  0.0, 2.0
amp2, mu2, sig2 = 1.5, 20.0, 6.0

params = torch.tensor([amp1, mu1, sig1, amp2, mu2, sig2], dtype=torch.float32)
params = params.reshape(1, 6, 1, 1)

curve = criterion.reconstruct_gaussians(params)[0, :, 0, 0]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(x_axis.numpy(), curve.numpy(), color='C0', label='reconstructed mixture')
ax.axvline(mu1, color='C1', ls='--', lw=1, label=f'mu1 = {mu1:g}')
ax.axvline(mu2, color='C2', ls='--', lw=1, label=f'mu2 = {mu2:g}')
ax.set_xlabel('elevation x [m]')
ax.set_ylabel('reflectivity [a.u.]')
ax.set_title('Two-component Gaussian reconstruction')
ax.legend(loc='upper right', frameon=False)
fig.tight_layout()
plt.show()


## Sweep one parameter at a time

To confirm each parameter has the intended effect we vary amplitude, centre,
and width independently and overlay the resulting single-Gaussian curves.

In [ ]:
def single_curve(amp, mu, sig):
    p = torch.tensor([amp, mu, sig, 0.0, 0.0, 1.0], dtype=torch.float32).reshape(1, 6, 1, 1)
    return criterion.reconstruct_gaussians(p)[0, :, 0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=False)
xs = x_axis.numpy()

for amp in [1.0, 2.0, 3.0]:
    axes[0].plot(xs, single_curve(amp, 10.0, 4.0), label=f'amp={amp:g}')
axes[0].set_title('amplitude sweep'); axes[0].legend(frameon=False)

for mu in [0.0, 10.0, 20.0]:
    axes[1].plot(xs, single_curve(2.0, mu, 4.0), label=f'mu={mu:g}')
axes[1].set_title('centre sweep'); axes[1].legend(frameon=False)

for sig in [2.0, 4.0, 8.0]:
    axes[2].plot(xs, single_curve(2.0, 10.0, sig), label=f'sigma={sig:g}')
axes[2].set_title('width sweep'); axes[2].legend(frameon=False)

for ax in axes:
    ax.set_xlabel('elevation x [m]')
axes[0].set_ylabel('reflectivity [a.u.]')
fig.tight_layout()
plt.show()


## Expected visual outcome

The two-component reconstruction should show a tall narrow peak at `mu1 = 0`
and a shorter broad peak at `mu2 = 20`. In the sweeps, amplitude scales peak
height linearly, the centre sweep slides the peak along the x-axis, and the
width sweep widens and lowers the peak while conserving its location. This
confirms `reconstruct_gaussians` implements the intended forward model.